# Model Monitoring Basics with Iris

A compact, hands-on introduction to detecting dataset drift for the Iris classifier used in this repository. We compare the original Iris reference sample with a controlled, shifted feature sample, then check whether classification quality changes.

The shifted inputs below are synthetic educational data, not live production observations.

## Setup
From the repository root install `python -m pip install -r requirements.txt` and select that environment as the notebook kernel.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

SEED = 42
FEATURES = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
data = load_iris(as_frame=True)
X = data.data.copy()
X.columns = FEATURES
y = data.target
X_train, X_reference, y_train, y_reference = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
model = RandomForestClassifier(n_estimators=200, random_state=SEED).fit(X_train, y_train)
print(f"Iris reference rows: {len(X_reference)}")
display(X_reference.head())

Iris reference rows: 45


,sepal_length,sepal_width,petal_length,petal_width
107,7.3,2.9,6.3,1.8
63,6.1,2.9,4.7,1.4
133,6.3,2.8,5.1,1.5
56,6.3,3.3,4.7,1.6
127,6.1,3.0,4.9,1.8


## Simulate a shifted incoming period
Increase two petal measurements while retaining labels for offline quality evaluation. This models covariate drift in the Iris dataset.

In [2]:
X_shifted = X_reference.copy()
X_shifted["petal_length"] += 1.0
X_shifted["petal_width"] += 0.4
reference_predictions = model.predict(X_reference)
shifted_predictions = model.predict(X_shifted)
quality = pd.DataFrame({
    "reference": [accuracy_score(y_reference, reference_predictions), f1_score(y_reference, reference_predictions, average="weighted")],
    "shifted": [accuracy_score(y_reference, shifted_predictions), f1_score(y_reference, shifted_predictions, average="weighted")],
}, index=["accuracy", "weighted F1"])
display(quality.round(3))

,reference,shifted
accuracy,0.911,0.667
weighted F1,0.911,0.630


## Detect numeric feature drift
PSI summarizes change in binned distribution; KS provides a two-sample statistical test. PSI cutoffs are heuristics and should be validated for the application.

In [3]:
def psi(reference, current, bins=10):
    reference, current = np.asarray(reference), np.asarray(current)
    edges = np.unique(np.quantile(reference, np.linspace(0, 1, bins + 1)))
    if len(edges) < 2:
        raise ValueError("Reference values must include at least two distinct values")
    edges[0], edges[-1] = -np.inf, np.inf
    ref_share = np.histogram(reference, bins=edges)[0] / len(reference)
    cur_share = np.histogram(current, bins=edges)[0] / len(current)
    eps = 1e-6
    return float(np.sum((cur_share-ref_share) * np.log((cur_share+eps)/(ref_share+eps))))

rows = []
for feature in FEATURES:
    ks = ks_2samp(X_reference[feature], X_shifted[feature])
    rows.append({"feature": feature, "PSI": psi(X_reference[feature], X_shifted[feature]), "KS p-value": ks.pvalue})
display(pd.DataFrame(rows).set_index("feature").round(4))

,PSI,KS p-value
feature,,
sepal_length,0.0000,1.000
sepal_width,0.0000,1.000
petal_length,6.2960,0.003
petal_width,5.7573,0.013


## Track a stream in windows
Stable windows are followed by shifted windows; compare every window to the reference.

In [4]:
window_results = []
for window_id in range(4):
    window = X_reference.sample(n=50, replace=True, random_state=SEED + window_id).reset_index(drop=True)
    if window_id >= 2:
        window["petal_length"] += 1.0
        window["petal_width"] += 0.4
    for feature in FEATURES:
        window_results.append({"window": window_id + 1, "feature": feature, "PSI": psi(X_reference[feature], window[feature])})
display(pd.DataFrame(window_results).pivot(index="window", columns="feature", values="PSI").round(3))

feature,petal_length,petal_width,sepal_length,sepal_width
window,,,,
1,0.141,0.036,0.256,0.143
2,0.119,0.110,0.182,0.110
3,6.559,6.042,0.159,0.255
4,6.402,7.196,0.246,0.234


## Connect to the monitored Iris service
The repository API exposes request counts, latency, predictions, and rolling feature PSI at `/metrics`. Run the Docker Compose stack as described in the README, then inspect Prometheus at <http://localhost:9090> and Grafana at <http://localhost:3000>. The notebook simulation does not emit live service or host-resource telemetry.